# 01 — Environment Setup & Pre-trained Models

**Lab 2 — Group 4 | CIFAR-10**

Bước 1–3 của đề: thiết lập môi trường, tải VGG16/DenseNet121 và khảo sát kiến trúc.

Cấu hình được đặt trực tiếp trong cell code đầu tiên của notebook. `review` đọc kết quả lịch sử; `full` chạy mới. Mỗi notebook có thể mở bằng kernel riêng.

In [1]:
from pathlib import Path
import sys, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs').is_dir() and (p/'src').is_dir()), None)
assert ROOT is not None, 'Hãy mở notebook trong thư mục dự án đã giải nén.'
sys.path.insert(0, str(ROOT))
# Cấu hình trực tiếp trong notebook (không cần file cấu hình riêng).
MODE = "review"  # review: kết quả gốc; full: chạy thực tế
RUN_ID = "run_01"  # notebook 04–06 phải dùng cùng tên khi chạy full
DEVICE_OVERRIDE = "auto"  # auto / cpu / cuda
CPU_THREADS = 1
LIVE = MODE == "full"
assert MODE in ("review", "full")
DOWNLOAD = True
import io, json, random, hashlib, time, gc
from pathlib import Path
from datetime import datetime
from uuid import uuid4

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 42
CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck"]
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD = np.array([0.229, 0.224, 0.225])

DATA_DIR = ROOT / 'data'
SESSION_DIR = ROOT / 'outputs' / 'notebook_runs' / RUN_ID
FIG_DIR = SESSION_DIR / 'figures'
SAVE_FIGS = LIVE

def save_fig(name):
    if SAVE_FIGS:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        plt.savefig(FIG_DIR / name, dpi=150, bbox_inches="tight")

random.seed(SEED)
np.random.seed(SEED)
if LIVE:
    import torch
    import torchvision
    from torch import nn
    from torch.utils.data import DataLoader, Subset, random_split
    from torchvision import transforms, models
    from PIL import Image
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    from torch.utils.tensorboard import SummaryWriter
    import torch.nn.functional as F

    def set_seed(seed):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

    set_seed(SEED)
    DEVICE = torch.device(("cuda" if torch.cuda.is_available() else "cpu")
                          if DEVICE_OVERRIDE == "auto" else DEVICE_OVERRIDE)
    if DEVICE.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA không khả dụng. Chọn CPU hoặc sửa môi trường.")
    if DEVICE.type == "cpu":
        torch.set_num_threads(CPU_THREADS)
    print("PyTorch:", torch.__version__, "| Torchvision:", torchvision.__version__)
    print("Device:", DEVICE)
    print("Data:", DATA_DIR, "| Output mới:", SESSION_DIR)
else:
    print("Review: không cần import PyTorch, không tạo file training.")

Review: không cần import PyTorch, không tạo file training.


## Bước 1. Môi trường
Cài PyTorch/Torchvision phù hợp với CPU hoặc CUDA trước. Sau đó cài `requirements.txt` từ thư mục gốc. Không cần cài lại mỗi khi chạy notebook.

In [2]:
import platform
print('Python:',platform.python_version())
print('Project:',ROOT)
print('Mode:',MODE)
if LIVE:
    print('CUDA available:',torch.cuda.is_available())
else:
    print('Review không kiểm tra GPU / tải weights; full sẽ kiểm tra môi trường PyTorch.')

Python: 3.13.5
Project: /mnt/data/lab2_monitoring_work/Lab2_Group4_Complete
Mode: review
Review không kiểm tra GPU / tải weights; full sẽ kiểm tra môi trường PyTorch.


## 2. Load a Pre-trained Model — Tải mô hình pretrained
- VGG16 và DenseNet121 được huấn luyện trước trên ImageNet (1.000 lớp).
- Dùng `weights=..._Weights.DEFAULT`, tương đương mục đích của `pretrained=True` trong slide (API torchvision hiện dùng weights enum).
- Hàm `load_pretrained` cũng cho phép tạo kiến trúc không tải weights khi phục hồi checkpoint cuối.

In [3]:
def load_pretrained(name):
    if name == 'vgg16':
        return models.vgg16(weights=models.VGG16_Weights.DEFAULT)
    if name == 'densenet121':
        return models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
    raise ValueError(name)
print('VGG16 và DenseNet121: pretrained ImageNet, 1000 lớp.')

VGG16 và DenseNet121: pretrained ImageNet, 1000 lớp.


## 3. Explore the Model Architecture — Khảo sát kiến trúc
In cấu trúc mô hình, xác định lớp backbone, classifier và tổng số tham số. VGG16 gồm convolution/pooling và các lớp fully connected; DenseNet121 sử dụng dense blocks, pooling và classifier tuyến tính.

In [4]:
if LIVE:
    for name in ('vgg16','densenet121'):
        model = load_pretrained(name)
        print(name, model)
        print('Classifier:', model.classifier)
        print('Tổng parameters:', sum(p.numel() for p in model.parameters()))
        del model
        gc.collect()
else:
    display(pd.DataFrame([['VGG16','features + classifier','4096 → 1000'],['DenseNet121','dense blocks + classifier','1024 → 1000']], columns=['Model','Thành phần','Đầu ra gốc']))

,Model,Thành phần,Đầu ra gốc
0,VGG16,features + classifier,4096 → 1000
1,DenseNet121,dense blocks + classifier,1024 → 1000


**Kết luận:** chuyển từ 1000 lớp ImageNet sang 10 lớp CIFAR-10 ở notebook 02.